# Round 3: Synthetic Comment Generation (DG / SAG / SAGII)

This notebook generates synthetic comments for r/AskDocs OPs using three distinct
prompting strategies, each designed to test how generation method affects the
realism and diversity of synthetic discourse.

**Focus: AI vs. verified clinicians.** Analysis of the corpus showed that genuine
layperson responses are extremely rare on r/AskDocs (once OP self-posts and
bot/moderator messages are removed, only ~300 real layperson comments remain in the
entire corpus). The subreddit is built around verified professionals and moderators
remove layperson answers. We therefore focus the comparison on AI-generated vs. real
clinician comments, and all synthetic comments are generated in a clinician voice.

**Prompting strategies:**

1. **DG (Discourse Generation)**: A single prompt generates the entire thread of *n*
   clinician replies at once.

2. **SAG (Single Advice Generation)**: *n* independent API calls, each generating a
   single clinician response. Each call sees only the OP.

3. **SAGII (SAG with Incremental Information)**: *n* sequential API calls. Each call
   generates a single clinician response but also sees all previously generated
   responses in the thread so far.

**Key design choices:**
- *n* (the number of synthetic comments per OP) matches the number of real, non-self
  top-level comments in that thread.
- OP self-posts (comments by the OP on their own thread) are removed.
- OPs containing images are excluded (text-only threads).
- No keyword/topic filter: the OP pool is all image-free submissions with at least one
  real comment.
- 100 OPs are sampled for generation.
- Checkpointing supports resumption after interruption.
- Output is an Excel spreadsheet (1 tab per OP) for review.

## 1. Setup & Imports

In [1]:
# Install dependencies if needed
# !pip install google-genai tqdm openpyxl

In [2]:
import json
import os
import re
import random
import time
from getpass import getpass
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

from google import genai
from google.genai import types

In [3]:
# ---------- Configuration ----------
API_KEY = getpass("Enter your Google Gemini API key: ")
MODEL_ID = "gemini-2.5-pro"

# Paths  (notebook runs from src/, so corpora is at ../output/corpora)
CORPORA_DIR = Path("../output/corpora")
SUBMISSIONS_PATH = CORPORA_DIR / "submissions_corpus.jsonl"
COMMENTS_PATH = CORPORA_DIR / "comments_corpus.jsonl"
# Fresh checkpoint + output for the clinician-only run (no keyword filter)
CHECKPOINT_DIR = CORPORA_DIR / "checkpoints_round3_clinician"
OUTPUT_XLSX = CORPORA_DIR / "round3_clinician_synthetic_comments.xlsx"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Rate-limiting
DELAY_BETWEEN_CALLS = 1.0  # seconds between API calls

# Number of OPs to generate for
N_SAMPLE = 100

# Random seed for reproducibility
SEED = 42
random.seed(SEED)

print(f"Model: {MODEL_ID}")
print(f"Submissions: {SUBMISSIONS_PATH}")
print(f"Comments: {COMMENTS_PATH}")
print(f"Output spreadsheet: {OUTPUT_XLSX}")
print(f"Sample size: {N_SAMPLE} OPs")

Enter your Google Gemini API key:  ········


Model: gemini-2.5-pro
Submissions: ../output/corpora/submissions_corpus.jsonl
Comments: ../output/corpora/comments_corpus.jsonl
Output spreadsheet: ../output/corpora/round3_clinician_synthetic_comments.xlsx
Sample size: 100 OPs


In [4]:
# Initialize the Gemini client
client = genai.Client(api_key=API_KEY)
print("Gemini client initialized.")

Gemini client initialized.


## 2. Load Data & Filter OPs

In [5]:
# Load all submissions
submissions = []
with open(SUBMISSIONS_PATH) as f:
    for line in f:
        submissions.append(json.loads(line))

print(f"Loaded {len(submissions):,} total submissions")

# Build a lookup by submission id
submissions_by_id = {s["id"]: s for s in submissions}

Loaded 19,984 total submissions


In [6]:
# Load all comments and group by submission
#
# We also filter out "self-posts": top-level comments written by the OP
# themselves (i.e. the commenter's username matches the submission author).
# On r/AskDocs, people often post their question and then immediately add a
# follow-up comment underneath it. These are not genuine responses, and it turns
# out the vast majority of "layperson"-flaired comments are actually the OP
# self-posting, so removing them gives a much cleaner picture of real responders.
comments_by_submission = defaultdict(list)

n_total_toplevel = 0
n_self_posts = 0

with open(COMMENTS_PATH) as f:
    for line in f:
        comment = json.loads(line)
        # All comments in the corpus are top-level (parent_id starts with t3_)
        if comment.get("parent_id", "").startswith("t3_"):
            sub_id = comment["parent_id"][3:]
            n_total_toplevel += 1

            # Skip self-posts: comment author == OP (submission) author.
            # We match on username (the `author` field), which is present on both
            # comments and submissions. `is_submitter` is an equivalent flag.
            op_author = submissions_by_id.get(sub_id, {}).get("author")
            comment_author = comment.get("author")
            if comment_author and op_author and comment_author == op_author:
                n_self_posts += 1
                continue

            comments_by_submission[sub_id].append(comment)

print(f"Total top-level comments: {n_total_toplevel:,}")
print(f"Self-posts removed (comment author == OP): {n_self_posts:,}")
print(f"Remaining comments after filtering: {sum(len(v) for v in comments_by_submission.values()):,}")
print(f"Submissions with >=1 non-self comment: {len(comments_by_submission):,}")

Total top-level comments: 40,537
Self-posts removed (comment author == OP): 5,241
Remaining comments after filtering: 35,296
Submissions with >=1 non-self comment: 18,097


In [7]:
def has_real_comments(sub_id: str) -> bool:
    """Check whether a submission has at least one comment that isn't [removed] or [deleted]."""
    for c in comments_by_submission.get(sub_id, []):
        if c.get("body", "") not in ("[removed]", "[deleted]"):
            return True
    return False


def compute_flair_counts(sub_id: str) -> dict:
    """
    Compute clinician/layperson counts of the REAL comments directly from flairs.

    On r/AskDocs, 'flair' is a label next to a user's name indicating whether
    they are a verified medical professional (e.g. 'Physician', 'Registered Nurse')
    or an unverified user ('Layperson/not verified as healthcare professional').

    Note: self-posts by the OP have already been removed. These counts are kept only
    for reporting the composition of the real thread; synthetic generation is
    clinician-only regardless of this split.
    """
    comments = comments_by_submission.get(sub_id, [])
    total = len(comments)
    layperson = 0
    for c in comments:
        flair = c.get("author_flair_text", "") or ""
        if "layperson" in flair.lower():
            layperson += 1
    clinician = total - layperson
    return {"total": total, "clinician": clinician, "layperson": layperson}


# Detect OPs that contain images. All posts in the corpus are text ("self") posts,
# but many embed or link images. We flag a submission as having an image if:
#   (a) it has a Reddit-generated 'preview' object (only present when there's an image), or
#   (b) its body contains an image host URL or image file extension.
IMG_URL_PATTERN = re.compile(
    r'(i\.redd\.it|preview\.redd\.it|imgur\.com|\.jpg|\.jpeg|\.png|\.gif|\.webp|\.heic)',
    re.IGNORECASE
)

def has_image(submission: dict) -> bool:
    """Return True if the submission appears to contain or link an image."""
    if submission.get("preview"):
        return True
    if IMG_URL_PATTERN.search(submission.get("selftext", "") or ""):
        return True
    return False


# No keyword/topic filter. The OP pool is all image-free submissions that have at
# least one real (non-self) comment.
eligible_submissions = [
    s for s in submissions
    if not has_image(s)
    and has_real_comments(s['id'])
]

# Attach real-comment counts to each submission for downstream use
for s in eligible_submissions:
    counts = compute_flair_counts(s['id'])
    s['_flair_total'] = counts['total']
    s['_flair_clinician'] = counts['clinician']
    s['_flair_layperson'] = counts['layperson']

print(f"Eligible OPs (image-free, with >=1 real comment): {len(eligible_submissions):,}")
print(f"\nSample titles:")
for s in eligible_submissions[:5]:
    print(f"  [{s['_flair_total']} real comment(s)] {s['title']}")

Eligible OPs (image-free, with >=1 real comment): 12,464

Sample titles:
  [5 real comment(s)] How would I go about getting an amputation that is not necessary?
  [1 real comment(s)] Iron BW results
  [3 real comment(s)] TIA? Not sure what happened.
  [1 real comment(s)] gallstones & caffeine
  [1 real comment(s)] How much of a calorie deficit is too much? 15AFAB


In [8]:
# Select N_SAMPLE random OPs from the eligible set
sample_submissions = random.sample(eligible_submissions, min(N_SAMPLE, len(eligible_submissions)))

print(f"Selected {len(sample_submissions)} OPs for generation:")
print()

# Show real comment-count distribution for the sample
import statistics
sample_totals = [s['_flair_total'] for s in sample_submissions]

print(f"Real comments per OP across {len(sample_submissions)} OPs:")
print(f"  mean={statistics.mean(sample_totals):.1f}, median={statistics.median(sample_totals)}, range=[{min(sample_totals)}, {max(sample_totals)}]")
print(f"  Total real comments in sample: {sum(sample_totals)}")
print(f"\nEach OP will get {sum(sample_totals)} synthetic clinician comments per strategy where")
print(f"the count matches the real thread size.")
print(f"Estimated API calls: DG={len(sample_submissions)}, SAG~={sum(sample_totals)}, SAGII~={sum(sample_totals)}, Total~={len(sample_submissions) + 2*sum(sample_totals)}")

Selected 100 OPs for generation:

Real comments per OP across 100 OPs:
  mean=1.5, median=1.0, range=[1, 11]
  Total real comments in sample: 151

Each OP will get 151 synthetic clinician comments per strategy where
the count matches the real thread size.
Estimated API calls: DG=100, SAG~=151, SAGII~=151, Total~=402


## 3. Comment Count Helper

All synthetic comments are generated in a clinician voice. For each OP we generate
*n* comments, where *n* is the number of real (non-self) top-level comments in the
thread. This helper just reads that count.

In [9]:
def n_comments_for(submission: dict) -> int:
    """Number of synthetic comments to generate for an OP = its real comment count."""
    return submission.get('_flair_total', 0)


# Quick test
test_sub = sample_submissions[0]
print(f"Test OP: {test_sub['title'][:60]}")
print(f"  Will generate {n_comments_for(test_sub)} clinician comment(s) per strategy")

Test OP: What is this on my babys hand male 19 weeks 5 days
  Will generate 1 clinician comment(s) per strategy


## 4. Shared Prompt Guidelines

All three strategies share the same behavioral guidelines for how comments should
read. We define them once as a constant and inject them into each strategy's prompt.

In [10]:
COMMENT_GUIDELINES = (
    "Important guidelines for the comments:\n"
    "- Keep comments brief and natural. Match the typical length of real Reddit "
    "comments (usually 1-4 sentences).\n"
    "- Layperson comments do NOT need to be medically accurate or provide correct "
    "medical advice.\n"
    "- Each comment can focus on just one particular aspect of the OP rather than "
    "responding to the entire post.\n"
    "- Clinicians should NOT introduce themselves by stating their role or credentials "
    "(e.g. do not start with 'I'm a doctor' or 'As a nurse').\n"
    "- Clinicians should NOT include disclaimers like 'It's important to consult a "
    "physician' or 'You will want to ask a doctor about this'.\n"
    "- Laypersons should NOT disclaim their lack of medical training "
    "(e.g. do not say 'I'm not a doctor but' or 'I have no medical background').\n"
)

print("Shared guidelines defined.")
print(COMMENT_GUIDELINES)

Shared guidelines defined.
Important guidelines for the comments:
- Keep comments brief and natural. Match the typical length of real Reddit comments (usually 1-4 sentences).
- Layperson comments do NOT need to be medically accurate or provide correct medical advice.
- Each comment can focus on just one particular aspect of the OP rather than responding to the entire post.
- Clinicians should NOT introduce themselves by stating their role or credentials (e.g. do not start with 'I'm a doctor' or 'As a nurse').
- Clinicians should NOT include disclaimers like 'It's important to consult a physician' or 'You will want to ask a doctor about this'.
- Laypersons should NOT disclaim their lack of medical training (e.g. do not say 'I'm not a doctor but' or 'I have no medical background').



## 5. Prompt Builders for Each Strategy

In [11]:
def build_dg_prompt(title: str, selftext: str, n: int) -> str:
    """
    DG (Discourse Generation): Single prompt that generates the entire thread of
    n clinician comments.
    """
    prompt = (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        f"Generate exactly {n} comments (no more, no fewer) in which each comment "
        "responds to the opening post and any subsequent comments prior to it, i.e. none "
        "of the comments should be threaded. Each comment should imitate the response "
        "of a clinician.\n\n"
        f"{COMMENT_GUIDELINES}\n"
        f"Return your response as a JSON array of exactly {n} objects, where each "
        "object has these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": \"clinician\"\n\n"
        f"Opening Post:\n"
        f"Title: {title}\n\n"
        f"{selftext}"
    )
    return prompt


def build_sag_prompt(title: str, selftext: str) -> str:
    """
    SAG (Single Advice Generation): Independent prompt for a single clinician comment.
    The model sees only the OP.
    """
    prompt = (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        "Generate exactly 1 comment that responds to the opening post. "
        "The comment should imitate the response of a clinician.\n\n"
        f"{COMMENT_GUIDELINES}\n"
        "Return your response as a JSON array containing exactly 1 object with these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": \"clinician\"\n\n"
        f"Opening Post:\n"
        f"Title: {title}\n\n"
        f"{selftext}"
    )
    return prompt


def build_sagii_prompt(title: str, selftext: str, prior_comments: list[dict]) -> str:
    """
    SAGII (SAG with Incremental Information): Prompt for a single clinician comment
    that also includes all previously generated comments in this thread.
    """
    if prior_comments:
        prior_text = "\n\nThe following comments have already been posted in response to this OP:\n\n"
        for i, pc in enumerate(prior_comments, 1):
            prior_text += f"Comment {i}: {pc['body']}\n\n"
        context_instruction = (
            "Your comment should respond to the opening post. You may also acknowledge "
            "or build on the existing comments, but do not simply repeat what has already "
            "been said."
        )
    else:
        prior_text = ""
        context_instruction = "Your comment should respond to the opening post."

    prompt = (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        f"Generate exactly 1 comment. {context_instruction} "
        "The comment should imitate the response of a clinician.\n\n"
        f"{COMMENT_GUIDELINES}\n"
        "Return your response as a JSON array containing exactly 1 object with these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": \"clinician\"\n\n"
        f"Opening Post:\n"
        f"Title: {title}\n\n"
        f"{selftext}"
        f"{prior_text}"
    )
    return prompt


print("Prompt builders defined (DG, SAG, SAGII) -- clinician-only.")

Prompt builders defined (DG, SAG, SAGII) -- clinician-only.


## 6. Gemini API Call Logic

In [12]:
def call_gemini(prompt: str, n_expected: int, retries: int = 3) -> list[dict]:
    """
    Send a prompt to Gemini and parse the JSON array response.
    Returns a list of comment dicts with 'body' and 'author_type' fields.
    Truncates to n_expected if the model returns more.
    """
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=1.0,
                ),
            )
            raw_text = response.text.strip()
            comments = json.loads(raw_text)

            if not isinstance(comments, list):
                raise ValueError(f"Expected a JSON array, got {type(comments).__name__}")

            validated = []
            for c in comments:
                validated.append({
                    "body": str(c.get("body", "")),
                    "author_type": str(c.get("author_type", "unknown")),
                })

            # Truncate if the model returned too many
            if len(validated) > n_expected:
                print(f"  Note: Model returned {len(validated)} comments, truncating to {n_expected}")
                validated = validated[:n_expected]

            return validated

        except json.JSONDecodeError as e:
            print(f"  JSON parse error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  API error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)

    print(f"  WARNING: All {retries} retries exhausted. Returning empty list.")
    return []


print("API call function defined.")

API call function defined.


## 7. Generation Functions for Each Strategy

In [13]:
def generate_dg(submission: dict) -> list[dict]:
    """
    DG strategy: one API call produces the full thread of n clinician comments.
    """
    title = submission.get("title", "")
    selftext = submission.get("selftext", "")
    n = n_comments_for(submission)

    prompt = build_dg_prompt(title, selftext, n)
    comments = call_gemini(prompt, n_expected=n)
    time.sleep(DELAY_BETWEEN_CALLS)
    return comments


def generate_sag(submission: dict) -> list[dict]:
    """
    SAG strategy: n independent API calls, each generating 1 clinician comment.
    """
    title = submission.get("title", "")
    selftext = submission.get("selftext", "")
    n = n_comments_for(submission)

    all_comments = []
    for _ in range(n):
        prompt = build_sag_prompt(title, selftext)
        result = call_gemini(prompt, n_expected=1)
        if result:
            all_comments.append(result[0])
        else:
            all_comments.append({"body": "", "author_type": "clinician"})
        time.sleep(DELAY_BETWEEN_CALLS)

    return all_comments


def generate_sagii(submission: dict) -> list[dict]:
    """
    SAGII strategy: n sequential API calls, each clinician comment seeing prior responses.
    """
    title = submission.get("title", "")
    selftext = submission.get("selftext", "")
    n = n_comments_for(submission)

    all_comments = []
    for _ in range(n):
        prompt = build_sagii_prompt(title, selftext, prior_comments=all_comments)
        result = call_gemini(prompt, n_expected=1)
        if result:
            all_comments.append(result[0])
        else:
            all_comments.append({"body": "", "author_type": "clinician"})
        time.sleep(DELAY_BETWEEN_CALLS)

    return all_comments


print("Generation functions defined (DG, SAG, SAGII) -- clinician-only.")

Generation functions defined (DG, SAG, SAGII) -- clinician-only.


## 8. Checkpoint Utilities

In [14]:
CHECKPOINT_FILE = CHECKPOINT_DIR / "round3_progress.jsonl"


def load_checkpoint() -> tuple[dict, set]:
    """Load previously completed results. Returns (results_by_id, completed_ids)."""
    results_by_id = {}
    completed_ids = set()

    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            for line in f:
                record = json.loads(line)
                sub_id = record["submission_id"]
                results_by_id[sub_id] = record
                completed_ids.add(sub_id)
        print(f"Resumed from checkpoint: {len(completed_ids):,} OPs already completed.")
    else:
        print("No checkpoint found. Starting from scratch.")

    return results_by_id, completed_ids


def append_checkpoint(record: dict):
    """Append a single completed record to the checkpoint file."""
    with open(CHECKPOINT_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")


print(f"Checkpoint file: {CHECKPOINT_FILE}")

Checkpoint file: ../output/corpora/checkpoints_round3_clinician/round3_progress.jsonl


## 9. Main Generation Loop

For each OP, we run all three strategies (DG, SAG, SAGII) and checkpoint the results.
If the notebook is interrupted and restarted, it picks up where it left off.

In [15]:
results_by_id, completed_ids = load_checkpoint()

remaining = [s for s in sample_submissions if s["id"] not in completed_ids]

print(f"Selected OPs: {len(sample_submissions)}")
print(f"Already completed: {len(sample_submissions) - len(remaining)}")
print(f"Remaining: {len(remaining)}")

No checkpoint found. Starting from scratch.
Selected OPs: 100
Already completed: 0
Remaining: 100


In [16]:
errors = []

for i, sub in enumerate(tqdm(remaining, desc="Generating (DG + SAG + SAGII)")):
    sub_id = sub["id"]
    n = n_comments_for(sub)

    if n == 0:
        continue

    try:
        # --- DG ---
        dg_comments = generate_dg(sub)

        # --- SAG ---
        sag_comments = generate_sag(sub)

        # --- SAGII ---
        sagii_comments = generate_sagii(sub)

        record = {
            "submission_id": sub_id,
            "n_comments": n,
            "dg": dg_comments,
            "sag": sag_comments,
            "sagii": sagii_comments,
        }

        results_by_id[sub_id] = record
        completed_ids.add(sub_id)
        append_checkpoint(record)

    except Exception as e:
        errors.append((sub_id, str(e)))
        print(f"\nError on submission {sub_id}: {e}")

print(f"\nGeneration complete. Processed {len(completed_ids):,} OPs.")
if errors:
    print(f"Errors encountered: {len(errors)}")
    for sub_id, err in errors[:10]:
        print(f"  {sub_id}: {err}")

Generating (DG + SAG + SAGII): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [1:14:09<00:00, 44.50s/it]


Generation complete. Processed 100 OPs.


## 10. Preview Results

In [17]:
# Preview first 3 OPs
for sub in sample_submissions[:3]:
    sub_id = sub["id"]
    record = results_by_id.get(sub_id)
    if not record:
        continue

    real = comments_by_submission.get(sub_id, [])
    clinician = sub.get('_flair_clinician', 0)
    layperson = sub.get('_flair_layperson', 0)
    total = clinician + layperson

    print(f"{'='*80}")
    print(f"OP: {sub['title']}")
    print(f"Real breakdown: {clinician} clinician, {layperson} layperson ({total} total)")
    print()

    print("-- Real comments --")
    for c in sorted(real, key=lambda x: x.get("created_utc", 0)):
        if c.get("body", "") in ("[removed]", "[deleted]"):
            continue
        flair = c.get("author_flair_text") or "No flair"
        print(f"  [{flair}] {c['body']}")
        print()

    for strategy_key, strategy_label in [("dg", "DG"), ("sag", "SAG"), ("sagii", "SAGII")]:
        print(f"-- {strategy_label} --")
        for c in record.get(strategy_key, []):
            print(f"  [{c['author_type']}] {c['body']}")
            print()

    print()

OP: What is this on my babys hand male 19 weeks 5 days
Real breakdown: 1 clinician, 0 layperson (1 total)

-- Real comments --
  [Physician] Could be a cafe au lait macule, could be an early hemangioma. Just have the doc take a look at the next visit.

-- DG --
  [clinician] It's always a good idea to look very closely with a bright light to ensure there isn't a fine hair or thread wrapped around it. This is called a hair tourniquet and can be easy to miss.

-- SAG --
  [clinician] This is very common in infants due to them keeping their fists clenched. Moisture and lint get trapped in the skin folds, causing irritation. Keep it clean and dry, and it should clear up.

-- SAGII --
  [clinician] This looks like a normal palmar crease that has become irritated, which is very common in infants. Keep it clean and dry, especially after baths.


OP: Getting a "second opinion" to calm my nerves I guess
Real breakdown: 1 clinician, 0 layperson (1 total)

-- Real comments --
  [Physician | Top C

## 11. Export to Spreadsheet

Each OP gets its own tab. Within each tab:
- The Opening Post (title + body + real breakdown)
- Real Comments section
- DG section
- SAG section
- SAGII section

In [18]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = Workbook()
wb.remove(wb.active)  # Remove default sheet

header_font = Font(bold=True, size=11, name="Arial")
body_font = Font(size=10, name="Arial")
title_font = Font(bold=True, size=12, name="Arial")
section_fill = PatternFill('solid', fgColor='D9E1F2')  # light blue
wrap_alignment = Alignment(wrap_text=True, vertical='top')
thin_border = Border(bottom=Side(style='thin', color='CCCCCC'))

strategy_labels = {
    "dg": "Synthetic: DG (Discourse Generation)",
    "sag": "Synthetic: SAG (Single Advice Generation)",
    "sagii": "Synthetic: SAGII (SAG + Incremental Info)",
}


def sanitize_sheet_name(name: str) -> str:
    """Remove characters that Excel does not allow in sheet names."""
    for ch in ['[', ']', '*', '?', '/', '\\', ':']:
        name = name.replace(ch, '')
    return name


for idx, sub in enumerate(tqdm(sample_submissions, desc="Building spreadsheet")):
    sub_id = sub["id"]
    record = results_by_id.get(sub_id)
    if not record:
        continue

    real = comments_by_submission.get(sub_id, [])
    real_sorted = sorted(real, key=lambda x: x.get("created_utc", 0))

    # Sheet name (Excel limits to 31 chars; strip illegal characters)
    short_title = sub.get("title", f"OP {idx+1}")[:26]
    sheet_name = sanitize_sheet_name(f"{idx+1}. {short_title}")
    if len(sheet_name) > 31:
        sheet_name = sheet_name[:31]
    ws = wb.create_sheet(title=sheet_name)

    ws.column_dimensions['A'].width = 18
    ws.column_dimensions['B'].width = 90

    row = 1

    # ---- Opening Post ----
    ws.cell(row=row, column=1, value="Opening Post").font = title_font
    row += 1
    ws.cell(row=row, column=1, value="Title:").font = header_font
    ws.cell(row=row, column=2, value=sub.get("title", "")).font = body_font
    ws.cell(row=row, column=2).alignment = wrap_alignment
    row += 1
    ws.cell(row=row, column=1, value="Body:").font = header_font
    ws.cell(row=row, column=2, value=sub.get("selftext", "")).font = body_font
    ws.cell(row=row, column=2).alignment = wrap_alignment
    row += 1

    clinician_count = sub.get('_flair_clinician', 0)
    layperson_count = sub.get('_flair_layperson', 0)
    total = clinician_count + layperson_count
    ws.cell(row=row, column=1, value="Real breakdown:").font = header_font
    ws.cell(row=row, column=2,
            value=f"{clinician_count} clinician, {layperson_count} layperson ({total} total)"
    ).font = body_font
    row += 2

    # ---- Real Comments ----
    ws.cell(row=row, column=1, value="Real Comments").font = title_font
    ws.cell(row=row, column=1).fill = section_fill
    ws.cell(row=row, column=2).fill = section_fill
    row += 1
    ws.cell(row=row, column=1, value="Author Type").font = header_font
    ws.cell(row=row, column=2, value="Comment").font = header_font
    row += 1
    for c in real_sorted:
        if c.get("body", "") in ("[removed]", "[deleted]"):
            continue
        flair = c.get("author_flair_text") or "No flair"
        ws.cell(row=row, column=1, value=flair).font = body_font
        ws.cell(row=row, column=1).alignment = wrap_alignment
        ws.cell(row=row, column=2, value=c.get("body", "")).font = body_font
        ws.cell(row=row, column=2).alignment = wrap_alignment
        ws.cell(row=row, column=1).border = thin_border
        ws.cell(row=row, column=2).border = thin_border
        row += 1
    row += 1

    # ---- Synthetic: DG, SAG, SAGII ----
    for strategy_key, label in strategy_labels.items():
        ws.cell(row=row, column=1, value=label).font = title_font
        ws.cell(row=row, column=1).fill = section_fill
        ws.cell(row=row, column=2).fill = section_fill
        row += 1
        ws.cell(row=row, column=1, value="Author Type").font = header_font
        ws.cell(row=row, column=2, value="Comment").font = header_font
        row += 1
        for c in record.get(strategy_key, []):
            ws.cell(row=row, column=1, value=c["author_type"]).font = body_font
            ws.cell(row=row, column=1).alignment = wrap_alignment
            ws.cell(row=row, column=2, value=c["body"]).font = body_font
            ws.cell(row=row, column=2).alignment = wrap_alignment
            ws.cell(row=row, column=1).border = thin_border
            ws.cell(row=row, column=2).border = thin_border
            row += 1
        row += 1

wb.save(OUTPUT_XLSX)
print(f"Spreadsheet saved to {OUTPUT_XLSX}")
print(f"Tabs: {wb.sheetnames}")

Building spreadsheet: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 825.02it/s]


Spreadsheet saved to ../output/corpora/round3_clinician_synthetic_comments.xlsx
Tabs: ['1. What is this on my babys h', '2. Getting a "second opinion"', '3. Middle ear infection, seco', '4. 34F on multiple meds start', '5. Dissolvable stitches in a ', '6. Is it hemochromatosis Or ', '7. Red Raised Painful Bump on', '8. What is it 27M', '9. Fever 3 days after surgica', '10. 30 year old male experienc', '11. Numbness after surgery (20', '12. My husband is showing sign', '13. Help with weed', '14. Nanotechnology in blood', '15. How do I know when or if I', '16. How do the relationships b', '17. Is it cancer', '18. I think my lungs are givin', '19. Is it safe to drive a car ', '20. Is this worth going to urg', '21. Breast Cancer, just parano', '22. Men’s health tests', '23. Depersonalization of multi', '24. Question about my appendix', '25. I took a year worth of vit', '26. 21M foul smelling nose lin', '27. Am I still contagious on d', '28. how bad is swallowing a ve', '29. Recurrent thrus

## 12. Summary Statistics

In [19]:
# Summary across all generated OPs
total_dg = 0
total_sag = 0
total_sagii = 0
total_real = 0

for sub in sample_submissions:
    sub_id = sub["id"]
    record = results_by_id.get(sub_id)
    if not record:
        continue

    real = [c for c in comments_by_submission.get(sub_id, [])
            if c.get("body", "") not in ("[removed]", "[deleted]")]

    total_real += len(real)
    total_dg += len(record.get("dg", []))
    total_sag += len(record.get("sag", []))
    total_sagii += len(record.get("sagii", []))

print(f"OPs processed: {len(completed_ids)}")
print(f"Total real comments: {total_real}")
print(f"Total DG comments: {total_dg}")
print(f"Total SAG comments: {total_sag}")
print(f"Total SAGII comments: {total_sagii}")
print(f"\nAvg comments per OP:")
n = len(completed_ids) or 1
print(f"  Real: {total_real/n:.1f}")
print(f"  DG:   {total_dg/n:.1f}")
print(f"  SAG:  {total_sag/n:.1f}")
print(f"  SAGII: {total_sagii/n:.1f}")

OPs processed: 100
Total real comments: 113
Total DG comments: 151
Total SAG comments: 151
Total SAGII comments: 151

Avg comments per OP:
  Real: 1.1
  DG:   1.5
  SAG:  1.5
  SAGII: 1.5
